In [1]:
import numpy as np
import os
from tqdm import tqdm

# Paths
TUPLES_PATH = r"D:\Machine Learning\BTP\Smart-Recommendation-System\quintuplet_tuples\quintuplet_tuples.npy"
EMBED_DIR   = r"D:\Machine Learning\BTP\Smart-Recommendation-System\embeddings"
IDS_DIR     = r"D:\Machine Learning\BTP\Smart-Recommendation-System\ids"

# 1. Load the "Required" universe
print("Loading tuples...")
tuples = np.load(TUPLES_PATH)
required_ids = set(tuples.flatten().astype(np.int64))
total_required = len(required_ids)
print(f"Total unique photo IDs required: {total_required:,}")

# 2. Prepare files for incremental writing
# We use memmap to create a file on disk that acts like an array
embed_output_path = "train_embeddings_2M.npy"
ids_output_path = "train_ids_2M.npy"

# Create the shell for the IDs first
final_ids_mapped = np.zeros(total_required, dtype=np.int64)
# Create the shell for Embeddings (Total Required x 2048)
# This creates a large file on disk immediately (approx 16GB)
final_embeds_mapped = np.lib.format.open_memmap(
    embed_output_path, mode='w+', dtype='float32', shape=(total_required, 2048)
)

# 3. Processing
id_files = sorted([f for f in os.listdir(IDS_DIR) if f.startswith("ids_chunk_")])
current_pos = 0

for id_file in tqdm(id_files, desc="Processing Chunks"):
    chunk_idx = id_file.split('_')[-1].replace('.npy', '')
    embed_file = f"embeddings_chunk_{chunk_idx}.npy"
    embed_path = os.path.join(EMBED_DIR, embed_file)
    
    if not os.path.exists(embed_path):
        continue
    
    # Load IDs chunk
    chunk_ids = np.load(os.path.join(IDS_DIR, id_file)).astype(np.int64)
    
    # Find matches
    mask = np.isin(chunk_ids, list(required_ids))
    num_matches = np.sum(mask)
    
    if num_matches > 0:
        # Load embeddings only for this chunk
        chunk_embeds = np.load(embed_path)
        
        # Calculate how many we are adding
        end_pos = current_pos + num_matches
        
        # Write directly to the DISK-mapped array (RAM stays low)
        final_ids_mapped[current_pos:end_pos] = chunk_ids[mask]
        final_embeds_mapped[current_pos:end_pos] = chunk_embeds[mask]
        
        current_pos = end_pos
        
        # Remove from required_ids set to speed up np.isin in future iterations
        # (Optional, but helps performance)
        # required_ids.difference_update(chunk_ids[mask])

# 4. Final Clean up
# Since current_pos might be slightly less than total_required if some IDs were missing
if current_pos < total_required:
    print(f"Note: Only {current_pos} of {total_required} IDs were found in chunks.")
    # Trim the files if necessary
    final_ids_mapped = final_ids_mapped[:current_pos]
    # For memmap, we just note the actual size to use in Notebook 7

np.save(ids_output_path, final_ids_mapped)
final_embeds_mapped.flush() # Ensure all data is written to disk
del final_embeds_mapped # Close the file handle

print(f"\nSuccess! Saved to {embed_output_path}")

Loading tuples...
Total unique photo IDs required: 2,851,129


Processing Chunks: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 768/768 [24:58<00:00,  1.95s/it]


Note: Only 2597860 of 2851129 IDs were found in chunks.

Success! Saved to train_embeddings_2M.npy
